# Séance 15 · Exercices — Les agents · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

**Niveau de la séance : ⭐⭐⭐ Avancé** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler :
1. Lis l'énoncé, puis code dans la cellule « À toi » (les variables à remplir sont déjà nommées).
2. Lance la cellule de vérification : ✅ c'est bon, ❌ on réessaie (l'indice est là pour ça).
3. Ouvre la solution **seulement après avoir essayé**.

Ce notebook tourne dans **Google Colab** : rien à installer. Pour exécuter une cellule : clique dedans puis `Maj + Entrée`.

Un nouvel agent, construit de zéro : **l'assistant de la médiathèque du lycée**. Il cherche un livre, donne les horaires, calcule une amende de retard, et réserve un livre (mais seulement si tu dis oui). Nouveauté par rapport à la leçon : le modèle demande ses outils en **JSON** (comme à la séance 13), par exemple `{"outil": "chercher_livre", "argument": "Dune"}`.

Les exercices s'enchaînent : chaque outil sert à l'agent des exercices suivants, fais-les dans l'ordre. Tous les exercices marchent en **mode démo** (`USE_MODEL = False`) : le faux modèle suit le protocole JSON dès que le prompt système le décrit. Ce qu'on vérifie, c'est **ton agent**, pas le talent du modèle.

## Préparation

Le catalogue de la médiathèque (`CATALOGUE`, 6 livres, données inventées), ses horaires (`HORAIRES`), la fonction `llm(messages)` (le vrai petit modèle, ou le faux en mode démo) et `verifier`. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import pandas as pd

# ---------- Les données de la médiathèque du lycée (inventées pour l'exercice) ----------
CATALOGUE = pd.DataFrame([
    {"titre": "Le Petit Prince", "auteur": "Antoine de Saint-Exupéry", "genre": "conte", "disponible": True, "retour": None},
    {"titre": "Dune", "auteur": "Frank Herbert", "genre": "science-fiction", "disponible": False, "retour": "2026-11-04"},
    {"titre": "Fahrenheit 451", "auteur": "Ray Bradbury", "genre": "science-fiction", "disponible": True, "retour": None},
    {"titre": "L'Étranger", "auteur": "Albert Camus", "genre": "roman", "disponible": False, "retour": "2026-10-28"},
    {"titre": "Persepolis", "auteur": "Marjane Satrapi", "genre": "bande dessinée", "disponible": True, "retour": None},
    {"titre": "Le Horla", "auteur": "Guy de Maupassant", "genre": "nouvelle", "disponible": True, "retour": None},
])
HORAIRES = {"lundi": "10 h - 17 h", "mardi": "10 h - 17 h", "mercredi": "9 h - 12 h", "jeudi": "10 h - 18 h", "vendredi": "10 h - 16 h"}
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]

# ---------- Mode démo : un faux LLM qui suit le protocole JSON quand le prompt système le décrit ----------
def llm_factice(messages):
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    question = next(m["content"] for m in messages if m["role"] == "user")
    ql = question.lower()
    if '"outil"' in systeme:
        resultats = [m["content"].split(":", 1)[1].split("\n")[0].strip()
                     for m in messages if m["role"] == "user" and m["content"].startswith("Résultat de l'outil")]
        if resultats:
            return "D'après mes outils : " + " ; ".join(resultats)
        titre = next((t for t in CATALOGUE["titre"] if t.lower() in ql), None)
        jour = next((j for j in JOURS if j in ql), None)
        nombre = re.search(r"\d+", ql)
        if titre and re.search(r"réserv", ql):
            return json.dumps({"outil": "reserver", "argument": titre}, ensure_ascii=False)
        if titre:
            return json.dumps({"outil": "chercher_livre", "argument": titre}, ensure_ascii=False)
        if jour and re.search(r"ouvert|horaire|heure", ql):
            return json.dumps({"outil": "horaires", "argument": jour}, ensure_ascii=False)
        if nombre and re.search(r"retard|amende", ql):
            return json.dumps({"outil": "amende", "argument": nombre.group(0)}, ensure_ascii=False)
        return "Bonjour ! Je suis l'assistant de la médiathèque du lycée."
    return "Je ne connais pas la médiathèque de ton lycée, mais elle a sûrement de bons livres."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------
if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen, ou une fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

print("Modèle :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)", "·", len(CATALOGUE), "livres au catalogue")

## Exercice 1 ⭐ · Lire un appel d'outil en JSON

Le modèle demande un outil avec une ligne de JSON : `{"outil": "chercher_livre", "argument": "Dune"}`. Écris `lire_appel(reponse)` qui renvoie
le couple `(outil, argument)`, ou `None` si la réponse n'est pas du JSON, ou si elle n'a pas de clé `"outil"`.

Résultat attendu sur les quatre essais : `("chercher_livre", "Dune")`, `("horaires", "mardi")`, `None`, `None`.

<details><summary>Indice</summary>

`json.loads` lève une erreur sur du texte qui n'est pas du JSON : entoure-le d'un `try / except`. Puis `if "outil" not in donnees: return None`.

</details>

In [ ]:
# À toi
def lire_appel(reponse):
    return None

essais = ['{"outil": "chercher_livre", "argument": "Dune"}', '{"outil": "horaires", "argument": "mardi"}',
          "Bonjour, je suis l'assistant !", '{"titre": "Dune"}']
for e in essais:
    print(f"{e!r:55} → {lire_appel(e)}")

In [ ]:
verifier("Exercice 1", [lire_appel(e) for e in essais] == [("chercher_livre", "Dune"), ("horaires", "mardi"), None, None])

<details><summary>Solution</summary>

```python
def lire_appel(reponse):
    try:
        donnees = json.loads(reponse)
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(donnees, dict) or "outil" not in donnees:
        return None
    return donnees["outil"], donnees.get("argument", "")

essais = ['{"outil": "chercher_livre", "argument": "Dune"}', '{"outil": "horaires", "argument": "mardi"}',
          "Bonjour, je suis l'assistant !", '{"titre": "Dune"}']
for e in essais:
    print(f"{e!r:55} → {lire_appel(e)}")
```

Avec du JSON, plus besoin d'expression régulière : le format est strict, et `json.loads` dit tout de suite si la réponse le respecte.

</details>

## Exercice 2 ⭐ · Le premier outil : chercher un livre

Écris `chercher_livre(titre)` qui cherche le livre dans `CATALOGUE` (majuscules et espaces ignorés) et renvoie :

- `"Dune (Frank Herbert) : emprunté, retour prévu le 2026-11-04."` s'il est emprunté,
- `"Persepolis (Marjane Satrapi) : disponible."` s'il est là,
- `"Aucun livre appelé X au catalogue."` sinon.

<details><summary>Indice</summary>

Le filtre : `CATALOGUE[CATALOGUE["titre"].str.lower() == titre.strip().lower()]`. Puis `if ligne.empty`, sinon `l = ligne.iloc[0]` et deux phrases selon `l["disponible"]`.

</details>

In [ ]:
# À toi
def chercher_livre(titre):
    return None

print(chercher_livre("dune"))
print(chercher_livre("Persepolis"))
print(chercher_livre("Harry Potter"))

In [ ]:
verifier("Exercice 2", chercher_livre(" dune ") == "Dune (Frank Herbert) : emprunté, retour prévu le 2026-11-04." and chercher_livre("Persepolis") == "Persepolis (Marjane Satrapi) : disponible." and chercher_livre("Harry Potter") == "Aucun livre appelé Harry Potter au catalogue.")

<details><summary>Solution</summary>

```python
def chercher_livre(titre):
    ligne = CATALOGUE[CATALOGUE["titre"].str.lower() == titre.strip().lower()]
    if ligne.empty:
        return f"Aucun livre appelé {titre.strip()} au catalogue."
    l = ligne.iloc[0]
    if l["disponible"]:
        return f"{l['titre']} ({l['auteur']}) : disponible."
    return f"{l['titre']} ({l['auteur']}) : emprunté, retour prévu le {l['retour']}."

print(chercher_livre("dune"))
print(chercher_livre("Persepolis"))
print(chercher_livre("Harry Potter"))
```

</details>

## Exercice 3 ⭐ · Un outil qui ne plante jamais

Écris `horaires(jour)` qui renvoie `"Le mardi : 10 h - 17 h."` pour un jour d'ouverture (majuscules ignorées), `"Le samedi : fermé."`
pour un vrai jour où la médiathèque est fermée, et `"Je ne connais pas le jour « X »."` pour tout le reste.
Un outil d'agent reçoit n'importe quoi : il ne doit **jamais** lever d'erreur.

<details><summary>Indice</summary>

`jour = jour.strip().lower()`, puis trois cas : `if jour in HORAIRES`, `elif jour in JOURS`, `else`.

</details>

In [ ]:
# À toi
def horaires(jour):
    return None

for j in ["Mardi", "samedi", "jeudi ", "demain"]:
    print(horaires(j))

In [ ]:
verifier("Exercice 3", horaires("Mardi") == "Le mardi : 10 h - 17 h." and horaires("samedi") == "Le samedi : fermé." and horaires("demain") == "Je ne connais pas le jour « demain »." and horaires("jeudi ") == "Le jeudi : 10 h - 18 h.")

<details><summary>Solution</summary>

```python
def horaires(jour):
    jour = jour.strip().lower()
    if jour in HORAIRES:
        return f"Le {jour} : {HORAIRES[jour]}."
    elif jour in JOURS:
        return f"Le {jour} : fermé."
    else:
        return f"Je ne connais pas le jour « {jour} »."

for j in ["Mardi", "samedi", "jeudi ", "demain"]:
    print(horaires(j))
```

« demain » est une vraie question d'utilisateur. Un meilleur outil calculerait le jour avec `datetime` : c'est une amélioration possible, l'important est de ne pas planter.

</details>

## Exercice 4 ⭐⭐ · Un outil qui vérifie ses arguments

Écris `amende(jours)` : la médiathèque compte **0,20 € par jour de retard**, au plus **5 €**. L'argument arrive en **texte** (le modèle l'écrit) :

- `amende("3")` → `"3 jours de retard : 0,60 €."` (avec une virgule) ;
- `amende("40")` → `"40 jours de retard : 5,00 €."` (le plafond) ;
- `amende("-2")`, `amende("trois")` → `"Nombre de jours invalide : -2."` (ou `trois`).

<details><summary>Indice</summary>

`int(jours.strip())` dans un `try` ; si ça rate ou si c'est négatif, le message d'erreur. Le montant : `min(n * 0.20, 5)`, écrit avec `f"{montant:.2f}".replace(".", ",")`.

</details>

In [ ]:
# À toi
def amende(jours):
    return None

for j in ["3", "40", "-2", "trois", "0"]:
    print(amende(j))

In [ ]:
verifier("Exercice 4", amende("3") == "3 jours de retard : 0,60 €." and amende("40") == "40 jours de retard : 5,00 €." and amende("-2") == "Nombre de jours invalide : -2." and amende("trois") == "Nombre de jours invalide : trois." and amende("0") == "0 jours de retard : 0,00 €.")

<details><summary>Solution</summary>

```python
def amende(jours):
    try:
        n = int(jours.strip())
    except ValueError:
        return f"Nombre de jours invalide : {jours.strip()}."
    if n < 0:
        return f"Nombre de jours invalide : {n}."
    montant = f"{min(n * 0.20, 5):.2f}".replace(".", ",")
    return f"{n} jours de retard : {montant} €."

for j in ["3", "40", "-2", "trois", "0"]:
    print(amende(j))
```

Le modèle peut écrire n'importe quoi dans l'argument : c'est l'outil qui fait la police. On n'exécute jamais une valeur qu'on n'a pas vérifiée.

</details>

## Exercice 5 ⭐⭐ · Le dispatcher

Les outils sont rangés dans `OUTILS` (nom → fonction). Écris `executer(outil, argument)` qui renvoie le résultat de l'outil, et qui ne plante jamais :

- outil inconnu → `"Outil inconnu : X. Outils disponibles : [...]"` ;
- outil qui lève une erreur → `"L'outil X a planté : <l'erreur>."`.

L'outil `outil_fragile` (fourni) plante exprès, pour tester.

<details><summary>Indice</summary>

`if outil not in OUTILS: return ...`, puis `try: return OUTILS[outil](argument)` et `except Exception as erreur: return f"L'outil {outil} a planté : {erreur}."`.

</details>

In [ ]:
# À toi
def outil_fragile(argument):
    raise ValueError("base de données injoignable")

OUTILS = {"chercher_livre": chercher_livre, "horaires": horaires, "amende": amende, "outil_fragile": outil_fragile}

def executer(outil, argument):
    return None

print(executer("horaires", "lundi"))
print(executer("voler", "Dune"))
print(executer("outil_fragile", "Dune"))

In [ ]:
verifier("Exercice 5", executer("horaires", "lundi") == "Le lundi : 10 h - 17 h." and executer("voler", "Dune").startswith("Outil inconnu : voler") and "chercher_livre" in executer("voler", "Dune") and executer("outil_fragile", "x") == "L'outil outil_fragile a planté : base de données injoignable.")

<details><summary>Solution</summary>

```python
def outil_fragile(argument):
    raise ValueError("base de données injoignable")

OUTILS = {"chercher_livre": chercher_livre, "horaires": horaires, "amende": amende, "outil_fragile": outil_fragile}

def executer(outil, argument):
    if outil not in OUTILS:
        return f"Outil inconnu : {outil}. Outils disponibles : {list(OUTILS)}"
    try:
        return OUTILS[outil](argument)
    except Exception as erreur:
        return f"L'outil {outil} a planté : {erreur}."

print(executer("horaires", "lundi"))
print(executer("voler", "Dune"))
print(executer("outil_fragile", "Dune"))
```

Une erreur renvoyée en texte, le modèle peut la lire et réessayer autrement. Une erreur Python, elle, arrête tout le programme.

</details>

## Exercice 6 ⭐⭐ · Décrire les outils au modèle

Le modèle ne connaît que les outils qu'on lui décrit. Écris `prompt_systeme(descriptions)` qui renvoie le prompt système complet :

- la première ligne : `"Tu es l'assistant de la médiathèque du lycée. Tes outils :"` ;
- une ligne par outil : `- nom(argument) : description.` ;
- puis la règle : `Pour utiliser un outil, réponds UNIQUEMENT une ligne de JSON, par exemple {"outil": "chercher_livre", "argument": "Dune"}. Sinon, réponds en une phrase.`

Les lignes sont séparées par des retours à la ligne.

<details><summary>Indice</summary>

`lignes = [f"- {nom}({arg}) : {desc}." for nom, (arg, desc) in descriptions.items()]`, puis `"\n".join([ENTETE] + lignes + [REGLE])`.

</details>

In [ ]:
# À toi
DESCRIPTIONS = {
    "chercher_livre": ("titre", "dit si un livre est au catalogue et s'il est disponible"),
    "horaires": ("jour", "les horaires d'ouverture d'un jour de la semaine"),
    "amende": ("jours", "l'amende pour un nombre de jours de retard"),
}
ENTETE = "Tu es l'assistant de la médiathèque du lycée. Tes outils :"
REGLE = 'Pour utiliser un outil, réponds UNIQUEMENT une ligne de JSON, par exemple {"outil": "chercher_livre", "argument": "Dune"}. Sinon, réponds en une phrase.'

def prompt_systeme(descriptions):
    return None

SYSTEME = prompt_systeme(DESCRIPTIONS)
print(SYSTEME)

In [ ]:
verifier("Exercice 6", SYSTEME is not None and SYSTEME.split("\n")[0] == ENTETE and SYSTEME.split("\n")[2] == "- horaires(jour) : les horaires d'ouverture d'un jour de la semaine." and SYSTEME.endswith(REGLE) and len(SYSTEME.split("\n")) == 5)

<details><summary>Solution</summary>

```python
DESCRIPTIONS = {
    "chercher_livre": ("titre", "dit si un livre est au catalogue et s'il est disponible"),
    "horaires": ("jour", "les horaires d'ouverture d'un jour de la semaine"),
    "amende": ("jours", "l'amende pour un nombre de jours de retard"),
}
ENTETE = "Tu es l'assistant de la médiathèque du lycée. Tes outils :"
REGLE = 'Pour utiliser un outil, réponds UNIQUEMENT une ligne de JSON, par exemple {"outil": "chercher_livre", "argument": "Dune"}. Sinon, réponds en une phrase.'

def prompt_systeme(descriptions):
    lignes = [f"- {nom}({arg}) : {desc}." for nom, (arg, desc) in descriptions.items()]
    return "\n".join([ENTETE] + lignes + [REGLE])

SYSTEME = prompt_systeme(DESCRIPTIONS)
print(SYSTEME)
```

Générer le prompt à partir d'un dictionnaire évite l'oubli classique : ajouter un outil à `OUTILS` mais pas dans le texte que lit le modèle.

</details>

## Exercice 7 ⭐⭐ · La boucle de l'agent

Écris `agent(question, max_etapes=3)` : le modèle répond ; si sa réponse est un appel d'outil (`lire_appel`), on l'exécute (`executer`),
on ajoute sa réponse et le résultat aux messages (`"Résultat de l'outil : …"`), et on recommence ; sinon, c'est la réponse finale.
Au-delà de `max_etapes`, on abandonne avec un message.

Le prompt système `SYSTEME` est redonné dans la cellule, pour que l'exercice marche seul.

<details><summary>Indice</summary>

La structure de la leçon, avec `lire_appel` à la place de l'expression régulière. Le message de résultat doit commencer par `"Résultat de l'outil"` : c'est ce que le faux modèle reconnaît.

</details>

In [ ]:
# À toi
SYSTEME = "\n".join(["Tu es l'assistant de la médiathèque du lycée. Tes outils :",
                      "- chercher_livre(titre) : dit si un livre est disponible.",
                      "- horaires(jour) : les horaires d'un jour.",
                      "- amende(jours) : l'amende pour des jours de retard.",
                      'Pour utiliser un outil, réponds UNIQUEMENT une ligne de JSON, par exemple {"outil": "chercher_livre", "argument": "Dune"}. Sinon, réponds en une phrase.'])

def agent(question, max_etapes=3):
    return "à compléter"

print(agent("Est-ce que Dune est disponible ?"))
print(agent("La médiathèque est ouverte le mercredi ?"))
print(agent("Bonjour !"))

In [ ]:
verifier("Exercice 7", "2026-11-04" in agent("Est-ce que Dune est disponible ?") and "9 h - 12 h" in agent("La médiathèque est ouverte le mercredi ?") and "assistant" in agent("Bonjour !").lower())

<details><summary>Solution</summary>

```python
SYSTEME = "\n".join(["Tu es l'assistant de la médiathèque du lycée. Tes outils :",
                      "- chercher_livre(titre) : dit si un livre est disponible.",
                      "- horaires(jour) : les horaires d'un jour.",
                      "- amende(jours) : l'amende pour des jours de retard.",
                      'Pour utiliser un outil, réponds UNIQUEMENT une ligne de JSON, par exemple {"outil": "chercher_livre", "argument": "Dune"}. Sinon, réponds en une phrase.'])

def agent(question, max_etapes=3):
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    for etape in range(max_etapes):
        reponse = llm(messages, max_new_tokens=80, temperature=0)
        appel = lire_appel(reponse)
        if appel is None:
            return reponse
        resultat = executer(*appel)
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {max_etapes} étapes."

print(agent("Est-ce que Dune est disponible ?"))
print(agent("La médiathèque est ouverte le mercredi ?"))
print(agent("Bonjour !"))
```

Si tu passes `USE_MODEL = True`, le petit modèle oubliera parfois le JSON : c'est le même raté que dans la leçon, avec un autre format.

</details>

## Exercice 8 ⭐⭐⭐ · La trace de l'agent

Pour comprendre ce que fait un agent, on garde sa **trace**. Écris `agent_trace(question)` qui fait la même chose que `agent`, mais renvoie
un couple `(reponse_finale, trace)`, où `trace` est la liste des `(outil, argument, resultat)` exécutés, dans l'ordre.

Résultat attendu pour « J'ai rendu un livre avec 12 jours de retard, je dois combien ? » : une trace d'un seul appel, à `amende` avec `"12"`.

<details><summary>Indice</summary>

Reprends ta boucle de l'exercice 7, avec une liste `trace = []` créée au début, et `trace.append((outil, argument, resultat))` après chaque exécution.

</details>

In [ ]:
# À toi
def agent_trace(question, max_etapes=3):
    return "à compléter", []

reponse, trace = agent_trace("J'ai rendu un livre avec 12 jours de retard, je dois combien ?")
print(reponse)
print(trace)

In [ ]:
verifier("Exercice 8", (lambda r_t: r_t[1] == [("amende", "12", "12 jours de retard : 2,40 €.")] and "2,40" in r_t[0])(agent_trace("J'ai rendu un livre avec 12 jours de retard, je dois combien ?")))

<details><summary>Solution</summary>

```python
SYSTEME = "\n".join(["Tu es l'assistant de la médiathèque du lycée. Tes outils :",
                      "- chercher_livre(titre) : dit si un livre est disponible.",
                      "- horaires(jour) : les horaires d'un jour.",
                      "- amende(jours) : l'amende pour des jours de retard.",
                      'Pour utiliser un outil, réponds UNIQUEMENT une ligne de JSON, par exemple {"outil": "chercher_livre", "argument": "Dune"}. Sinon, réponds en une phrase.'])

def agent_trace(question, max_etapes=3):
    trace = []
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    for etape in range(max_etapes):
        reponse = llm(messages, max_new_tokens=80, temperature=0)
        appel = lire_appel(reponse)
        if appel is None:
            return reponse, trace
        resultat = executer(*appel)
        trace.append((appel[0], appel[1], resultat))
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {max_etapes} étapes.", trace

reponse, trace = agent_trace("J'ai rendu un livre avec 12 jours de retard, je dois combien ?")
print(reponse)
print(trace)
```

La trace, c'est ce qu'on regarde quand un agent répond faux : a-t-il choisi le mauvais outil, mal écrit l'argument, ou mal lu le résultat ?

</details>

## Exercice 9 ⭐⭐⭐ · Le banc de test

Six questions, et l'outil qu'on attend pour chacune (`None` = aucun outil). Écris `outil_choisi(question)` qui renvoie l'outil que le modèle
demande **au premier tour** (ou `None`), puis `score` : le nombre de bons choix sur le banc.

En mode démo, le score attendu est 6 / 6 ; avec le vrai petit modèle, il sera plus bas, et c'est instructif.

<details><summary>Indice</summary>

Un seul appel à `llm` avec `SYSTEME` et la question, puis `lire_appel` : `appel[0] if appel else None`. Le score : `sum(outil_choisi(q) == attendu for q, attendu in BANC)`.

</details>

In [ ]:
# À toi
BANC = [
    ("Est-ce que Persepolis est disponible ?", "chercher_livre"),
    ("Vous ouvrez à quelle heure le jeudi ?", "horaires"),
    ("5 jours de retard, ça fait combien d'amende ?", "amende"),
    ("Merci beaucoup !", None),
    ("Le Horla est là ?", "chercher_livre"),
    ("Quels sont les horaires du lundi ?", "horaires"),
]

def outil_choisi(question):
    return None

score = None
print(f"Score : {score} / {len(BANC)}")

In [ ]:
verifier("Exercice 9", [outil_choisi(q) for q, _ in BANC] == [a for _, a in BANC] and score == len(BANC))

<details><summary>Solution</summary>

```python
BANC = [
    ("Est-ce que Persepolis est disponible ?", "chercher_livre"),
    ("Vous ouvrez à quelle heure le jeudi ?", "horaires"),
    ("5 jours de retard, ça fait combien d'amende ?", "amende"),
    ("Merci beaucoup !", None),
    ("Le Horla est là ?", "chercher_livre"),
    ("Quels sont les horaires du lundi ?", "horaires"),
]

def outil_choisi(question):
    reponse = llm([{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}], max_new_tokens=80, temperature=0)
    appel = lire_appel(reponse)
    return appel[0] if appel else None

score = sum(outil_choisi(q) == attendu for q, attendu in BANC)
print(f"Score : {score} / {len(BANC)}")
```

La question « Merci beaucoup ! » est la plus importante du banc : un agent qui appelle un outil pour tout gaspille du temps, et parfois de l'argent.

</details>

## Exercice 10 ⭐⭐⭐ · Un humain dans la boucle

Nouvel outil : `reserver(titre)`, qui réserve un livre (fourni, il remplit la liste `RESERVATIONS`). Réserver est une **action** : l'agent ne doit
pas la faire sans l'accord de l'utilisateur. Écris `executer_prudent(outil, argument, confirmer)` :

- pour un outil de `ACTIONS` (ici `reserver`), on appelle d'abord `confirmer(question)`, une fonction qui renvoie `True` ou `False` ;
- si la réponse est `False`, on n'exécute rien et on renvoie `"Action annulée : reserver(Dune) n'a pas été confirmée."` ;
- sinon, et pour tous les autres outils, on exécute normalement avec `executer`.

<details><summary>Indice</summary>

La question posée à l'utilisateur peut être `f"L'agent veut faire {outil}({argument}). D'accord ?"`. Pour tester sans clavier, la vérification passe des fonctions `confirmer` qui disent toujours oui, ou toujours non.

</details>

In [ ]:
# À toi
RESERVATIONS = []

def reserver(titre):
    RESERVATIONS.append(titre.strip())
    return f"{titre.strip()} est réservé pour toi."

OUTILS["reserver"] = reserver
ACTIONS = {"reserver"}

def executer_prudent(outil, argument, confirmer):
    return executer(outil, argument)

print(executer_prudent("reserver", "Dune", confirmer=lambda q: False))
print(RESERVATIONS)

In [ ]:
verifier("Exercice 10", (lambda: (RESERVATIONS.clear(), executer_prudent("reserver", "Dune", lambda q: False) == "Action annulée : reserver(Dune) n'a pas été confirmée." and RESERVATIONS == [] and executer_prudent("reserver", "Dune", lambda q: True) == "Dune est réservé pour toi." and RESERVATIONS == ["Dune"] and executer_prudent("horaires", "lundi", lambda q: False) == "Le lundi : 10 h - 17 h.")[1]))

<details><summary>Solution</summary>

```python
RESERVATIONS = []

def reserver(titre):
    RESERVATIONS.append(titre.strip())
    return f"{titre.strip()} est réservé pour toi."

OUTILS["reserver"] = reserver
ACTIONS = {"reserver"}

def executer_prudent(outil, argument, confirmer):
    if outil in ACTIONS and not confirmer(f"L'agent veut faire {outil}({argument}). D'accord ?"):
        return f"Action annulée : {outil}({argument}) n'a pas été confirmée."
    return executer(outil, argument)

print(executer_prudent("reserver", "Dune", confirmer=lambda q: False))
print(RESERVATIONS)
```

Dans une vraie appli, `confirmer` affiche un bouton « Oui / Non ». Chercher un livre n'engage à rien ; réserver, payer, envoyer, effacer : on demande toujours.

</details>

## Exercice 11 ⭐⭐⭐ · Quiz : les bonnes pratiques avec l'IA

Pour chaque situation, `True` (bonne idée) ou `False` (mauvaise idée), dans la liste `mes_reponses`, dans l'ordre :

1. Je laisse un agent répondre seul aux e-mails de mes profs, sans relire.
2. Je demande à l'IA de m'expliquer une notion de maths, puis je refais l'exercice seul.
3. Je colle dans un chatbot le carnet de santé de mon petit frère pour qu'il me résume ses vaccins.
4. Je vérifie dans le catalogue de la médiathèque un titre de livre que l'IA m'a conseillé avant d'aller le chercher.
5. Je donne à mon agent un outil qui peut supprimer des fichiers, sans confirmation, « pour gagner du temps ».
6. Je compare la réponse de l'IA avec mon manuel quand elle me donne une date historique.

<details><summary>Indice</summary>

Les quatre réflexes de la leçon : vérifier, ne pas donner de données personnelles, apprendre plutôt que copier, rester aux commandes des actions.

</details>

In [ ]:
# À toi
mes_reponses = [None, None, None, None, None, None]

In [ ]:
verifier("Exercice 11", mes_reponses == [False, True, False, True, False, True])

<details><summary>Solution</summary>

```python
mes_reponses = [False, True, False, True, False, True]
```

La 3 piège souvent : un carnet de santé, c'est une donnée personnelle, et celle de quelqu'un d'autre. On ne l'envoie pas à un service en ligne.

</details>

## Exercice 12 ⭐⭐⭐ · Défi : l'assistant de la médiathèque, complet

Assemble tout dans `assistant(question, confirmer)` : la boucle de l'exercice 7, la trace de l'exercice 8, et l'exécution prudente de l'exercice 10
(avec l'outil `reserver` décrit dans le prompt système). Elle renvoie `(reponse_finale, trace)`.

La vérification pose trois questions : un livre disponible, une réservation **refusée** par l'utilisateur, et une réservation **acceptée**.

<details><summary>Indice</summary>

Ajoute à `SYSTEME` la ligne `- reserver(titre) : réserve un livre (demande toujours l'accord de l'utilisateur).`, avant la règle. Dans la boucle, remplace `executer(*appel)` par `executer_prudent(appel[0], appel[1], confirmer)`.

</details>

In [ ]:
# À toi
def assistant(question, confirmer, max_etapes=3):
    return "à compléter", []

print(assistant("Est-ce que Fahrenheit 451 est disponible ?", confirmer=lambda q: True))
print(assistant("Réserve-moi Le Petit Prince", confirmer=lambda q: False))

In [ ]:
verifier("Exercice 12", (lambda: (RESERVATIONS.clear(), "disponible" in assistant("Est-ce que Fahrenheit 451 est disponible ?", lambda q: True)[0] and assistant("Réserve-moi Le Petit Prince", lambda q: False)[1][0][0] == "reserver" and RESERVATIONS == [] and "réservé" in assistant("Réserve-moi Le Petit Prince", lambda q: True)[0] and RESERVATIONS == ["Le Petit Prince"])[1]))

<details><summary>Solution</summary>

```python
SYSTEME_COMPLET = SYSTEME.replace("Pour utiliser un outil", "- reserver(titre) : réserve un livre (demande toujours l'accord de l'utilisateur).\nPour utiliser un outil")

def assistant(question, confirmer, max_etapes=3):
    messages = [{"role": "system", "content": SYSTEME_COMPLET}, {"role": "user", "content": question}]
    trace = []
    for etape in range(max_etapes):
        reponse = llm(messages, max_new_tokens=80, temperature=0)
        appel = lire_appel(reponse)
        if appel is None:
            return reponse, trace
        resultat = executer_prudent(appel[0], appel[1], confirmer)
        trace.append((appel[0], appel[1], resultat))
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {max_etapes} étapes.", trace

print(assistant("Est-ce que Fahrenheit 451 est disponible ?", confirmer=lambda q: True))
print(assistant("Réserve-moi Le Petit Prince", confirmer=lambda q: False))
```

Tu viens d'écrire, en petit, ce que font les agents des grands services : des outils sûrs, un protocole strict, une trace, et un humain qui garde la main sur les actions.

</details>

## Bravo !

Tu sais construire un agent de zéro : des outils qui ne plantent jamais et vérifient leurs arguments, un protocole d'appel en JSON, une boucle bornée,
une trace pour comprendre ce qui s'est passé, un banc de test pour le mesurer, et un humain dans la boucle pour les actions.

In [ ]:
reussis = sum(1 for ok in resultats.values() if ok)
print(f"{reussis} exercice(s) réussi(s) sur {len(resultats)}")
for nom, ok in resultats.items():
    print("  ", "✅" if ok else "❌", nom)